# 07 — Iteration 4: term-frequency saturation

BM25's `k1` controls how much repeated query words increase a passage score. Lowering it makes those repeated occurrences contribute less quickly. This candidate changes only `k1` from 1.5 to 0.8.

The diagnostic loop selected the untried change because wrong-section retrieval still dominates. This notebook reviews saved results and tries the saved model; it does not start another improvement iteration.

In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
from improve import read_log
from retrieval import BM25Retriever, load_inputs

run = next(r for r in reversed(read_log(ROOT / 'experiments/loop_log.jsonl')) if r['iteration'] == 4)
print('Changed:', run['changes'])
print('Reason:', run['proposal']['reason'])
print('Accepted:', run['decision']['accepted'])
rows = []
for label, metrics in run['metrics'].items():
    at5 = next(m for m in metrics['retrieval'] if m['k'] == 5)
    rows.append(dict(approach=label, precision_at_5=at5['precision'], recall_at_5=at5['recall'],
                     benchmark_precision=at5['benchmark_precision'], benchmark_recall=at5['benchmark_recall'],
                     correct_abstention=metrics['correct_abstention'], false_abstention=metrics['false_abstention']))
display(pd.DataFrame(rows))
display(pd.Series(run['decision']['checks'], name='Passed'))

Changed: {'k1': {'before': 1.5, 'after': 0.8}}
Reason: Earlier term-frequency saturation may reduce the influence of repeated words.
Accepted: False


,approach,precision_at_5,recall_at_5,benchmark_precision,benchmark_recall,correct_abstention,false_abstention
0,baseline,0.017433,0.181647,0.017594,0.192427,0.0,0.0
1,reference,0.017433,0.181647,0.017594,0.192427,0.0,0.0
2,candidate,0.015726,0.161617,0.016032,0.172396,0.0,0.0


recall_gain                False
precision                  False
answerable_abstention       True
unanswerable_abstention     True
latency                     True
Name: Passed, dtype: bool

## Try the saved candidate

Change `which` to `reference_config` to use the original baseline. Edit the question for an interview demonstration. Both configurations use the same local corpus and require no external model calls.

In [3]:
which = 'candidate_config'
_, documents, _ = load_inputs(ROOT)
model = BM25Retriever(documents, run[which])
question = 'Under the Fox and Glu Mobile wireless content license, when does the term expire?'
hits = model.search(question, k=5)
if hits:
    display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
else:
    print('No relevant passage found.')

,score,file_path,start,end,text
0,48.753816,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,27.270624,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
2,25.590264,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."
3,18.129007,cuad/FLOTEKINDUSTRIESINCCN_05_09_2019-EX-10.1-...,1600,2600,"ting Principles, consistently applied), plus a..."
4,17.667796,cuad/KINGPHARMACEUTICALSINC_08_09_2006-EX-10.1...,65600,66600,"grants to Depomed a non-assignable, non-subli..."


## Explain this step

“We tested whether repeated words were dominating BM25 rankings. Reducing their influence lowered unique-character recall@5 from 18.16% to 16.16%, and precision from 1.74% to 1.57%. We rejected the change and kept the baseline.”

Four distinct iterations are complete. These parameter trials have not produced a qualifying improvement; we should now examine a more direct cause of wrong-section retrieval, rather than assume additional small parameter changes will work. Separating the contract-identification wording from the clause question is a possible next hypothesis, not a measured result.

The revised target is seven iterations including later LLM trials. A good non-LLM comparison and functioning core come first; expand that target if necessary rather than move to LLMs with unresolved core failures. Both abstention rates and the fixed acceptance policy remain important. Test performance has not been measured.